# Principal Component Analysis (PCA)

PCA is a **dimensionality reduction** technique that projects data onto orthogonal axes (principal components) of maximum variance.

It is used for visualization, noise reduction, and feature extraction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Dataset

Using the **Wine** dataset (13 features) for dimensionality reduction.

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(data.feature_names)}")
print(f"Classes: {list(data.target_names)}")

## 2. Data Preprocessing

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled data shape: {X_scaled.shape}")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0).round(2)}")
print(f"Std  (should be ~1): {X_scaled.std(axis=0).round(2)}")

## 3. Apply PCA

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var)+1), explained_var, alpha=0.7, label='Individual')
axes[0].step(range(1, len(explained_var)+1), cumulative_var, where='mid', color='r', label='Cumulative')
axes[0].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance by Component')
axes[0].legend()

pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})')
axes[1].set_title('Data Projected onto First 2 PCs')
plt.colorbar(scatter, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"\nVariance explained by first 2 PCs: {sum(pca_2d.explained_variance_ratio_):.2%}")
for i, (var, cum) in enumerate(zip(explained_var, cumulative_var)):
    print(f"  PC{i+1}: {var:.4f} (cumulative: {cum:.4f})")

## 4. Hyperparameter Exploration

In [ ]:
param_grid = {
    'n_components': [2, 3, 4, 5, 0.90, 0.95, 0.99],
    'whiten': [True, False],
    'svd_solver': ['auto', 'full', 'arpack', 'randomized']
}

results = []
for n_comp in [2, 3, 4, 5]:
    for whiten in [True, False]:
        pca = PCA(n_components=n_comp, whiten=whiten)
        X_t = pca.fit_transform(X_scaled)
        total_var = sum(pca.explained_variance_ratio_)
        results.append({
            'n_components': n_comp,
            'whiten': whiten,
            'explained_variance': total_var,
            'reconstruction_error': np.mean((X_scaled - pca.inverse_transform(X_t))**2)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 5. Optimal PCA

In [ ]:
pca_95 = PCA(n_components=0.95)
X_reduced = pca_95.fit_transform(X_scaled)

print(f"Original features: {X_scaled.shape[1]}")
print(f"Reduced features:  {X_reduced.shape[1]}")
print(f"Variance retained: {sum(pca_95.explained_variance_ratio_):.2%}")

plt.figure(figsize=(8, 6))
components = pd.DataFrame(
    pca_95.components_[:2],
    columns=data.feature_names,
    index=['PC1', 'PC2']
)
sns.heatmap(components, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Loadings (PC1 and PC2)')
plt.tight_layout()
plt.show()